In [ ]:
#import library
import os, re, time, warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import csv
from pypdf import PdfReader
import cv2
from PIL import Image
import pytesseract
import fitz

ModuleNotFoundError: No module named 'pandas'

In [23]:
warnings.filterwarnings('ignore')

In [24]:
#change the directory
os.chdir('../../')
# vary fy that is that working or not
print("New working directory:", os.getcwd())

New working directory: d:\WASIM\project\ai_resume_job_matcher


In [26]:
#NLP / Text
import nltk
nltk.download('stopwords', quiet=-True)
nltk.download('punkt', quiet=True)
from nltk.corpus import stopwords

AttributeError: Module 'scipy' has no attribute '_lib'

In [11]:
#feature extraction
from sklearn.feature_extraction.text import TfidfTransformer

AttributeError: Module 'scipy' has no attribute '_lib'

In [12]:
#model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.pipeline import Pipeline

AttributeError: Module 'scipy' has no attribute '_lib'

In [13]:
#start extracting the data from pdf and shift them into a csv file

input_folder = 'data/raw/resumes/resumes_pdf'
output_csv = 'data/raw/data.csv'

In [19]:
def preprocess_image_for_ocr(img):
    # Convert to grayscale
    gray_img = cv2.cvtColor(np.array(img), cv2.COLOR_RGB2GRAY)

    # Binarization (thresholding) using OTSU
    _, binary_img = cv2.threshold(gray_img, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

    # Denoising
    denoised_img = cv2.medianBlur(binary_img, 3)

    return Image.fromarray(denoised_img)

In [20]:
def extract_text_from_pdf_ocr(pdf_path):
    """Extract text from a PDF file using OCR with preprocessing"""
    try:
        doc = fitz.open(pdf_path)
        ocr_text = []

        for i, page in enumerate(doc):
            # Render page to an image with high DPI (zoom=3)
            pix = page.get_pixmap(matrix=fitz.Matrix(3, 3))
            img_bytes = pix.tobytes("png")
            pil_img = Image.open(io.BytesIO(img_bytes))

            # Preprocess the image for better OCR accuracy
            processed_img = preprocess_image_for_ocr(pil_img)

            # Perform OCR
            text = pytesseract.image_to_string(processed_img)
            if text.strip():
                ocr_text.append(text.strip())

        doc.close()
        return "\n".join(ocr_text).strip()
    except Exception as e:
        print(f"Error reading {pdf_path}: {e}")
        return ""

In [21]:
data_rows = []
# Walk through all folders and files
for root, dirs, files in os.walk(input_folder):
    for file in files:
        if file.lower().endswith('.pdf'):
            # The direct parent folder name acts as the category header
            category = os.path.basename(root)
            pdf_path = os.path.join(root, file)

            print(f'Extracting text via OCR from: {category} --> {file}')

            # Extract text using OCR
            raw_text = extract_text_from_pdf_ocr(pdf_path)

            # Append rows (Category, Filename, Plain Text)
            data_rows.append([category, file, raw_text])

# Create a DataFrame to store and view the results
df_extracted = pd.DataFrame(data_rows, columns=['Category', 'Filename', 'Plain Text'])
display(df_extracted.head())


,Category,Filename,Plain Text


In [16]:
# Create the folders if they don't exist yet
#os.makedirs(os.path.dirname(output_csv), exist_ok=True)


#write data to csv file
with open(output_csv, mode='w', encoding='utf-8', newline='') as csv_file:
    writer = csv.writer(csv_file)
    #write the headers
    writer.writerow(["Category", "Filename", "Resume_Text"])
    #write the data rows
    writer.writerows(data_row)

print(f"\n Success! Saved clean text data to '{output_csv}'")


 Success! Saved clean text data to 'data/raw/data.csv'
